# 怎样使用预训练 Transformer 处理文本？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

输入形状 torch.Size([6, 24])
{
  "baseline_accuracy": 0.8333333333333334,
  "transformer_accuracy": 1.0,
  "format_valid_rate": 1.0,
  "generation_seconds": 0.131,
  "examples": 6,
  "model": "google/flan-t5-small",
  "elapsed_seconds": 9.422
}


## 1. 补全：解析边界
检查大小写、句点和未知输出。

In [2]:
assert parse_label(' Positive. ') == 1
assert parse_label('negative') == 0
assert parse_label('I am not certain') == -1
print('通过')

通过


## 2. 修改：观察补齐
两句不同长度输入一批，核对短句的真实长度较小。

In [3]:
b = tokenizer(['good', 'This is a much longer sentence about a product'], padding=True, return_tensors='pt')
assert b['attention_mask'][0].sum() < b['attention_mask'][1].sum()
ids = b['input_ids'][0][b['attention_mask'][0].bool()]
assert len(ids) == int(b['attention_mask'][0].sum())
print(tokenizer.decode(ids, skip_special_tokens=True))

good


## 3. 迁移：资料内抽取地点
让模型从资料抽取 Room 204，保留真实输出，检查是否命中人工答案。

In [4]:
inp = tokenizer('Context: The workshop takes place in Room 204. Question: In which room is the workshop? Answer briefly.', return_tensors='pt')
with torch.inference_mode():
    out = model.generate(**inp, max_new_tokens=10, do_sample=False)
answer = tokenizer.decode(out[0], skip_special_tokens=True)
assert isinstance(answer, str) and len(answer) > 0
print('真实输出', answer, '包含标准答案', '204' in answer)

真实输出 Room 204 包含标准答案 True
